# Lending Club Data Modeling

This notebook loads cleaned datasets and preprocessing artifacts from the data wrangling stage.

It provides:
- Training of multiple classification models (Logistic Regression and XGBoost)
- Comparison between fundamental and full feature sets
- Proper encoding and preprocessing pipelines for modeling
- Time-based validation and test evaluation
- Performance evaluation using ROC-AUC, PR-AUC, Brier score, and F1-score
- Threshold optimization based on validation data
- Export of trained models, encoders, predictions, and evaluation summaries for visualization

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    precision_recall_curve,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

from xgboost import XGBClassifier

In [2]:
# Load artifacts from wrangling step

ARTIFACT_DIR = Path("artifacts/cleaning_only")
MODEL_DIR = Path("artifacts/models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

CLEANED_OUTPUTS_PATH = ARTIFACT_DIR / "cleaned_outputs.joblib"
META_PATH = ARTIFACT_DIR / "meta.joblib"

if not CLEANED_OUTPUTS_PATH.exists():
    raise FileNotFoundError(f"Missing: {CLEANED_OUTPUTS_PATH}")
if not META_PATH.exists():
    raise FileNotFoundError(f"Missing: {META_PATH}")

meta = joblib.load(META_PATH)
cleaned_outputs = joblib.load(CLEANED_OUTPUTS_PATH)

print("Loaded wrangling artifacts.")
print("Meta:", meta)
print("Target note:", meta.get("target_definition_note", "N/A"))

Loaded wrangling artifacts.
Meta: {'snapshot_date': '2018-12-31', 'horizon_months': 12, 'split_years': {'train_end_year': 2015, 'val_year': 2016, 'test_year': 2017}, 'fundamental_drop': ['grade', 'sub_grade', 'int_rate', 'installment'], 'non_default_final_statuses': ['Does not meet the credit policy. Status: Fully Paid', 'Fully Paid'], 'active_nondefault_statuses': ['Current', 'In Grace Period', 'Late (16-30 days)', 'Late (31-120 days)'], 'default_statuses': ['Charged Off', 'Default', 'Does not meet the credit policy. Status: Charged Off'], 'target_definition_note': 'Default timing is approximated using last_pymnt_d because explicit default event timestamps are not included in the selected raw columns.'}
Target note: Default timing is approximated using last_pymnt_d because explicit default event timestamps are not included in the selected raw columns.


In [3]:
#  Unpack cleaned datasets + labels
X_train_fund_clean = cleaned_outputs["X_train_fund_clean"]
X_val_fund_clean = cleaned_outputs["X_val_fund_clean"]
X_test_fund_clean = cleaned_outputs["X_test_fund_clean"]

X_train_fund_xgb = cleaned_outputs["X_train_fund_xgb"]
X_val_fund_xgb = cleaned_outputs["X_val_fund_xgb"]
X_test_fund_xgb = cleaned_outputs["X_test_fund_xgb"]

X_train_full_xgb = cleaned_outputs["X_train_full_xgb"]
X_val_full_xgb = cleaned_outputs["X_val_full_xgb"]
X_test_full_xgb = cleaned_outputs["X_test_full_xgb"]

y_train_f = np.asarray(cleaned_outputs["y_train_f"]).astype(int)
y_val_f = np.asarray(cleaned_outputs["y_val_f"]).astype(int)
y_test_f = np.asarray(cleaned_outputs["y_test_f"]).astype(int)

y_train_full = np.asarray(cleaned_outputs["y_train_full"]).astype(int)
y_val_full = np.asarray(cleaned_outputs["y_val_full"]).astype(int)
y_test_full = np.asarray(cleaned_outputs["y_test_full"]).astype(int)

print("Shapes:")
print("  Fund logistic:", X_train_fund_clean.shape, X_val_fund_clean.shape, X_test_fund_clean.shape)
print("  Fund XGB:", X_train_fund_xgb.shape, X_val_fund_xgb.shape, X_test_fund_xgb.shape)
print("  Full XGB:", X_train_full_xgb.shape, X_val_full_xgb.shape, X_test_full_xgb.shape)
print("  Labels:", y_train_f.shape, y_val_f.shape, y_test_f.shape)

Shapes:
  Fund logistic: (886770, 49) (433890, 49) (442979, 49)
  Fund XGB: (886770, 49) (433890, 49) (442979, 49)
  Full XGB: (886770, 57) (433890, 57) (442979, 57)
  Labels: (886770,) (433890,) (442979,)


In [4]:
# Utilities (encoding + metrics + threshold)
def _build_ohe():
    # sklearn compatibility across versions
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)


def encode_categoricals(X_train, X_val, X_test, scale_numeric: bool):
    X_train = X_train.copy()
    X_val = X_val.copy()
    X_test = X_test.copy()

    cat_cols = X_train.select_dtypes(include=["object", "string", "category"]).columns.tolist()
    num_cols = [c for c in X_train.columns if c not in cat_cols]

    # normalize categorical missing values to np.nan
    for d in (X_train, X_val, X_test):
        for c in cat_cols:
            col = d[c].astype("object")
            d[c] = col.where(pd.notna(col), np.nan)

    if scale_numeric:
        num_pipe = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ])
    else:
        num_pipe = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median"))
        ])

    cat_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="__missing__")),
        ("ohe", _build_ohe())
    ])

    encoder = ColumnTransformer(
        transformers=[
            ("num", num_pipe, num_cols),
            ("cat", cat_pipe, cat_cols),
        ],
        remainder="drop"
    )

    X_train_enc = encoder.fit_transform(X_train)
    X_val_enc = encoder.transform(X_val)
    X_test_enc = encoder.transform(X_test)

    return encoder, X_train_enc, X_val_enc, X_test_enc, num_cols, cat_cols


def eval_scores(y_true, p):
    return {
        "roc_auc": roc_auc_score(y_true, p),
        "pr_auc": average_precision_score(y_true, p),
        "brier": brier_score_loss(y_true, p),
    }


def best_threshold_by_f1(y_true, p):
    precision, recall, thresholds = precision_recall_curve(y_true, p)
    if len(thresholds) == 0:
        return 0.50, 0.0
    f1_vals = 2 * precision[:-1] * recall[:-1] / np.clip(precision[:-1] + recall[:-1], 1e-12, None)
    idx = int(np.argmax(f1_vals))
    return float(thresholds[idx]), float(f1_vals[idx])


def fit_xgb_with_early_stopping(model, X_train, y_train, X_val, y_val, rounds=60):
    # Works across xgboost versions
    try:
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            verbose=False,
            early_stopping_rounds=rounds
        )
    except TypeError:
        model.set_params(early_stopping_rounds=rounds)
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            verbose=False
        )
    return model

In [5]:
# Encode the 3 model datasets
# Logistic (fundamental): scale numerics
enc_fund_log, X_train_fund_log_enc, X_val_fund_log_enc, X_test_fund_log_enc, num_f_log, cat_f_log = encode_categoricals(
    X_train_fund_clean, X_val_fund_clean, X_test_fund_clean, scale_numeric=True
)

# XGB (fundamental): no scaling needed
enc_fund_xgb, X_train_fund_xgb_enc, X_val_fund_xgb_enc, X_test_fund_xgb_enc, num_f_xgb, cat_f_xgb = encode_categoricals(
    X_train_fund_xgb, X_val_fund_xgb, X_test_fund_xgb, scale_numeric=False
)

# XGB (full): no scaling needed
enc_full_xgb, X_train_full_xgb_enc, X_val_full_xgb_enc, X_test_full_xgb_enc, num_full_xgb, cat_full_xgb = encode_categoricals(
    X_train_full_xgb, X_val_full_xgb, X_test_full_xgb, scale_numeric=False
)

print("fund_log cats:", len(cat_f_log), "shape:", X_train_fund_log_enc.shape)
print("fund_xgb cats:", len(cat_f_xgb), "shape:", X_train_fund_xgb_enc.shape)
print("full_xgb cats:", len(cat_full_xgb), "shape:", X_train_full_xgb_enc.shape)

fund_log cats: 5 shape: (886770, 100)
fund_xgb cats: 5 shape: (886770, 100)
full_xgb cats: 7 shape: (886770, 139)


In [6]:
# Train the 3 models
# class imbalance helper for XGB
neg_f = int((y_train_f == 0).sum())
pos_f = int((y_train_f == 1).sum())
scale_pos_weight_f = neg_f / max(pos_f, 1)

neg_full = int((y_train_full == 0).sum())
pos_full = int((y_train_full == 1).sum())
scale_pos_weight_full = neg_full / max(pos_full, 1)

print("scale_pos_weight_f:", round(scale_pos_weight_f, 4))
print("scale_pos_weight_full:", round(scale_pos_weight_full, 4))

# 1) Logistic Regression (Fundamental)
log_fund = LogisticRegression(
    solver="saga",
    max_iter=800,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
log_fund.fit(X_train_fund_log_enc, y_train_f)

# 2) XGB (Fundamental)
xgb_fund = XGBClassifier(
    n_estimators=1200,
    learning_rate=0.03,
    max_depth=6,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    objective="binary:logistic",
    eval_metric="aucpr",
    random_state=42,
    n_jobs=-1,
    tree_method="hist",
    scale_pos_weight=scale_pos_weight_f
)
xgb_fund = fit_xgb_with_early_stopping(
    xgb_fund, X_train_fund_xgb_enc, y_train_f, X_val_fund_xgb_enc, y_val_f, rounds=60
)

# 3) XGB (Full)
xgb_full = XGBClassifier(
    n_estimators=1200,
    learning_rate=0.03,
    max_depth=6,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    objective="binary:logistic",
    eval_metric="aucpr",
    random_state=42,
    n_jobs=-1,
    tree_method="hist",
    scale_pos_weight=scale_pos_weight_full
)
xgb_full = fit_xgb_with_early_stopping(
    xgb_full, X_train_full_xgb_enc, y_train_full, X_val_full_xgb_enc, y_val_full, rounds=60
)

print("Training complete.")

scale_pos_weight_f: 17.0172
scale_pos_weight_full: 17.0172


/Users/alex._choo/anaconda3/envs/am126/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Training complete.


In [7]:
# Evaluate all models (val/test)
rows = []

# Logistic (Fundamental)
p_val_log = log_fund.predict_proba(X_val_fund_log_enc)[:, 1]
p_test_log = log_fund.predict_proba(X_test_fund_log_enc)[:, 1]
val_m = eval_scores(y_val_f, p_val_log)
test_m = eval_scores(y_test_f, p_test_log)
thr_log, f1_log = best_threshold_by_f1(y_val_f, p_val_log)
rows.append({
    "model": "Logistic (Fundamental)",
    **{f"val_{k}": v for k, v in val_m.items()},
    **{f"test_{k}": v for k, v in test_m.items()},
    "val_best_thr_f1": thr_log,
    "val_best_f1": f1_log,
})

# XGB (Fundamental)
p_val_fund = xgb_fund.predict_proba(X_val_fund_xgb_enc)[:, 1]
p_test_fund = xgb_fund.predict_proba(X_test_fund_xgb_enc)[:, 1]
val_m = eval_scores(y_val_f, p_val_fund)
test_m = eval_scores(y_test_f, p_test_fund)
thr_fund, f1_fund = best_threshold_by_f1(y_val_f, p_val_fund)
rows.append({
    "model": "XGB (Fundamental)",
    **{f"val_{k}": v for k, v in val_m.items()},
    **{f"test_{k}": v for k, v in test_m.items()},
    "val_best_thr_f1": thr_fund,
    "val_best_f1": f1_fund,
})

# XGB (Full)
p_val_full = xgb_full.predict_proba(X_val_full_xgb_enc)[:, 1]
p_test_full = xgb_full.predict_proba(X_test_full_xgb_enc)[:, 1]
val_m = eval_scores(y_val_full, p_val_full)
test_m = eval_scores(y_test_full, p_test_full)
thr_full, f1_full = best_threshold_by_f1(y_val_full, p_val_full)
rows.append({
    "model": "XGB (Full)",
    **{f"val_{k}": v for k, v in val_m.items()},
    **{f"test_{k}": v for k, v in test_m.items()},
    "val_best_thr_f1": thr_full,
    "val_best_f1": f1_full,
})

summary = pd.DataFrame(rows).sort_values("val_pr_auc", ascending=False).reset_index(drop=True)
display(summary)

,model,val_roc_auc,val_pr_auc,val_brier,test_roc_auc,test_pr_auc,test_brier,val_best_thr_f1,val_best_f1
0,XGB (Full),0.717834,0.156783,0.221452,0.712393,0.141002,0.216958,0.667160,0.231502
1,XGB (Fundamental),0.672272,0.126729,0.221798,0.667310,0.110971,0.214419,0.589701,0.195692
2,Logistic (Fundamental),0.654211,0.114931,0.233022,0.650372,0.100494,0.226325,0.576052,0.182348


In [8]:
# Thresholded test reports using validation-chosen thresholds
model_threshold_plan = [
    ("Logistic (Fundamental)", y_test_f, p_test_log, thr_log),
    ("XGB (Fundamental)", y_test_f, p_test_fund, thr_fund),
    ("XGB (Full)", y_test_full, p_test_full, thr_full),
]

for name, y_true, p_test, thr in model_threshold_plan:
    y_pred = (p_test >= thr).astype(int)
    print("=" * 90)
    print(f"{name} | threshold={thr:.4f}")
    print("Confusion matrix:")
    print(confusion_matrix(y_true, y_pred))
    print("Precision:", round(precision_score(y_true, y_pred, zero_division=0), 4),
          "Recall:", round(recall_score(y_true, y_pred, zero_division=0), 4),
          "F1:", round(f1_score(y_true, y_pred, zero_division=0), 4))
    print("Classification report:")
    print(classification_report(y_true, y_pred, digits=4))

Logistic (Fundamental) | threshold=0.5761
Confusion matrix:
[[326474  89243]
 [ 16758  10504]]
Precision: 0.1053 Recall: 0.3853 F1: 0.1654
Classification report:
              precision    recall  f1-score   support

           0     0.9512    0.7853    0.8603    415717
           1     0.1053    0.3853    0.1654     27262

    accuracy                         0.7607    442979
   macro avg     0.5282    0.5853    0.5129    442979
weighted avg     0.8991    0.7607    0.8176    442979

XGB (Fundamental) | threshold=0.5897
Confusion matrix:
[[338684  77033]
 [ 17194  10068]]
Precision: 0.1156 Recall: 0.3693 F1: 0.1761
Classification report:
              precision    recall  f1-score   support

           0     0.9517    0.8147    0.8779    415717
           1     0.1156    0.3693    0.1761     27262

    accuracy                         0.7873    442979
   macro avg     0.5336    0.5920    0.5270    442979
weighted avg     0.9002    0.7873    0.8347    442979

XGB (Full) | threshold=0.66

In [9]:
# Save trained models + encoders + summary in a bundle for easy loading in deployment or reference
trained_bundle = {
    "meta": meta,
    "summary": summary,

    # models
    "log_fund": log_fund,
    "xgb_fund": xgb_fund,
    "xgb_full": xgb_full,

    # encoders
    "enc_fund_log": enc_fund_log,
    "enc_fund_xgb": enc_fund_xgb,
    "enc_full_xgb": enc_full_xgb,

    # optional threshold info for deployment/reference
    "thresholds": {
        "log_fund_val_best_f1_thr": float(thr_log),
        "xgb_fund_val_best_f1_thr": float(thr_fund),
        "xgb_full_val_best_f1_thr": float(thr_full),
    }
}

joblib.dump(trained_bundle, MODEL_DIR / "trained_models_bundle.joblib", compress=3)
summary.to_csv(MODEL_DIR / "model_summary.csv", index=False)

print("Saved:")
print(" -", MODEL_DIR / "trained_models_bundle.joblib")
print(" -", MODEL_DIR / "model_summary.csv")

Saved:
 - artifacts/models/trained_models_bundle.joblib
 - artifacts/models/model_summary.csv
